# SHORE — Interactive Visualisation

This notebook generates a unit-sphere 6-block cubed-sphere mesh from scratch, then
inspects the result using the OOP API and PyVista's HTML backend.

**Prerequisites**
```bash
pip install shore[vis]   # adds pyvista
```

In [ ]:
import numpy as np
import pyvista as pv

from shore.mesh import Body, CubedSphereTopology, Mesh, Spacing1D
from shore.mesh import info_geo, check_geo, export_geo, export_surface
from shore.io.geo import read_geo
from shore.io.vtk import open_to_pyvista

pv.set_jupyter_backend('html')

## 0 — Generate a unit-sphere 6-block cubed-sphere mesh

Creates `sphere.stl`, projects onto a structured `(ni×nj)` surface layer
saved as `sphere_k0.npz`, then marches `nk` wall-normal layers to produce
6 `.geo` files. Skips any step whose output already exists.

In [ ]:
from pathlib import Path
import trimesh

NI, NJ, NK = 40, 60, 20
DS, GROWTH = 0.02, 1.10
THETA_CAP = np.pi / 6.0  # 30 deg; lands inside the C1 cap/equator pinning window

STL_PATH = Path('sphere.stl')
GEO_STEM = Path('sphere')

LABELS    = ['sub0', 'sub1', 'sub2', 'sub3', 'cap_north', 'cap_south']
GEO_PATHS = [Path(f'sphere_{lbl}.geo') for lbl in LABELS]

# --- STL ---
if not STL_PATH.exists():
    sphere = trimesh.creation.icosphere(subdivisions=4, radius=1.0)
    sphere.export(str(STL_PATH))
    print(f'Generated {STL_PATH}  ({len(sphere.faces)} triangles)')
else:
    print(f'{STL_PATH} already exists, skipping.')

# --- generate + march ---
if not GEO_PATHS[0].exists():
    m = Mesh.from_stl(
        STL_PATH,
        topology=CubedSphereTopology(),
        ni=NI, nj=NJ, nk=NK,
        theta_cap=THETA_CAP,
        spacing_k=Spacing1D(law='geometric', ds=DS, growth=GROWTH),
    )

    def _cb(k: int, jmin: float) -> None:
        print(f'  layer {k:3d}  jmin={jmin:.3e}', end='\r')

    m.march(ds=DS, growth=GROWTH, layer_callback=_cb)
    print()
    m.write_geo(GEO_STEM)
    for blk in m.blocks:
        print(f'  sphere_{blk.label}.geo  shape={blk.nodes.shape[:3]}')
else:
    print('Geo files already exist, skipping marching.')

## 1 — Inspect block metadata

In [ ]:
from shore.mesh import info_geo

for p in GEO_PATHS:
    d  = info_geo(p)
    ni, nj, nk = d['shape']
    print(f"{p.name:25s}  ni={ni}  nj={nj}  nk={nk}  "
          f"points={d['n_points']:,}  cells={d['n_cells']:,}")

## 2 — Jacobian check per block

In [ ]:
from shore.mesh import check_geo

for p in GEO_PATHS:
    result = check_geo(p)
    jmins  = result['jmins']
    status = 'OK' if result['ok'] else 'INVERTED'
    print(f"{p.name:25s}  global_jmin={min(jmins):.4e}  status={status}")

## 3 — Per-layer Jacobian profile

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3))

for p, lbl in zip(GEO_PATHS, LABELS):
    jmins = check_geo(p)['jmins']
    ax.plot(range(1, len(jmins) + 1), jmins, marker='o', ms=3, lw=1.2, label=lbl)

ax.axhline(0, color='red', lw=0.8, ls='--', label='inversion threshold')
ax.set_xlabel('layer k')
ax.set_ylabel('jmin')
ax.set_title('Minimum cell Jacobian per layer — all 6 blocks')
ax.legend(fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4 — Visualise one sub-block coloured by Jacobian

In [ ]:
from shore.mesh import view_geo_file

pl = view_geo_file(GEO_PATHS[0], colorby='Jacobian', jupyter=True)
pl.show()

## 5 — Visualise all 6 blocks as a MultiBlock in PyVista

In [ ]:
from shore.io.geo import read_geo
from shore.io.vtk import open_to_pyvista

pv.set_jupyter_backend('html')
pl = pv.Plotter()

for p in GEO_PATHS:
    grid = read_geo(p)
    sg   = open_to_pyvista(grid, with_jacobian=True)
    pl.add_mesh(sg, scalars='Jacobian', cmap='RdYlGn',
                show_edges=True, opacity=0.7)

pl.add_axes()
pl.add_bounding_box()
pl.show()

## 6 — Surface layer (k=0)

In [ ]:
from shore.mesh import view_surface_file

# Display surface metadata
from shore.mesh import info_surface
d = info_surface(NPZ_PATH)
print(f"Surface shape : {d['shape']}")
print(f"Points        : {d['n_points']:,}")

pl = view_surface_file(NPZ_PATH, colorby='radius', jupyter=True)
pl.show()

## 7 — Export to VTK for ParaView

In [ ]:
from shore.mesh import export_geo, export_surface
from shore.io.vtk import write_caps_vtm

# Export surface
export_surface(NPZ_PATH)
print(f'Wrote sphere_k0.vts')

# Export all 6 blocks as a single MultiBlock
blocks_data = [read_geo(p) for p in GEO_PATHS]
write_caps_vtm('sphere.vtm', blocks_data, with_jacobian=True)
print('Wrote sphere.vtm — open in ParaView.')

## 8 — Spacing: surface clustering and wall-normal distributions

In [ ]:
from shore.mesh import Spacing1D

# Geometric (default)
sp_geom  = Spacing1D(law='geometric', ds=DS, growth=GROWTH)
k_geom   = sp_geom.build_steps(NK)

# Tanh: cluster near wall within fixed total thickness
sp_tanh  = Spacing1D(law='tanh', beta=4.0, total_thickness=2.0)
k_tanh   = sp_tanh.build_steps(NK)

# Tanh2: cluster at both wall and far-field
sp_tanh2 = Spacing1D(law='tanh2', beta=3.0, total_thickness=2.0)
k_tanh2  = sp_tanh2.build_steps(NK)

print(f'Geometric — first: {k_geom[0]:.4f}, last: {k_geom[-1]:.4f}, total: {k_geom.sum():.3f}')
print(f'Tanh      — first: {k_tanh[0]:.4f}, last: {k_tanh[-1]:.4f}, total: {k_tanh.sum():.3f}')
print(f'Tanh2     — first: {k_tanh2[0]:.4f}, last: {k_tanh2[-1]:.4f}, total: {k_tanh2.sum():.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
layers  = np.arange(1, NK)
ax.semilogy(layers, k_geom,  'o-', ms=3, label='geometric')
ax.semilogy(layers, k_tanh,  's-', ms=3, label='tanh (wall cluster)')
ax.semilogy(layers, k_tanh2, '^-', ms=3, label='tanh2 (wall + far-field)')
ax.set_xlabel('layer k')
ax.set_ylabel('step size')
ax.set_title('Wall-normal step-size distributions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9 — Generate a mesh with tanh wall-normal spacing

In [ ]:
TANH_STEM = Path('sphere_tanh')
TANH_SUB0 = Path('sphere_tanh_sub0.geo')

if not TANH_SUB0.exists():
    sp_k = Spacing1D(law='tanh', beta=4.0, total_thickness=2.0)
    m_tanh = Mesh.from_stl(
        STL_PATH,
        topology=CubedSphereTopology(),
        ni=NI, nj=NJ, nk=NK,
        theta_cap=THETA_CAP,
        spacing_k=sp_k,
    )
    m_tanh.march(ds=DS, growth=GROWTH, steps=sp_k.build_steps(NK))
    m_tanh.write_geo(TANH_STEM)
    print('Generated sphere_tanh_*.geo files.')
else:
    print('sphere_tanh_sub0.geo already exists, skipping.')

# Compare Jacobian profiles: geometric vs tanh
jmins_geom = check_geo(GEO_PATHS[0])['jmins']
jmins_tanh = check_geo(TANH_SUB0)['jmins']

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, len(jmins_geom) + 1), jmins_geom, 'o-', ms=4, lw=1.5, label='geometric (sub0)')
ax.plot(range(1, len(jmins_tanh) + 1), jmins_tanh, 's-', ms=4, lw=1.5, label='tanh beta=4 (sub0)')
ax.axhline(0, color='red', lw=0.8, ls='--')
ax.set_xlabel('layer k')
ax.set_ylabel('jmin')
ax.set_title('Jacobian profile: geometric vs tanh spacing (sub0 block)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10 — Cap/equator C1 seam matching

The cap k=0 layer is built by gnomonic flat-square projection (a tangent-plane square radially projected onto the body). The equator pins its first/last meridional step to the cap's first inward step at every seam vertex, giving **C1 spacing continuity** at the seam.

Pinning is only stable when the cap and equator radial step sizes are within ~2x of each other. When they aren't, the equator falls back to its unpinned `i_spacing` law and emits a `UserWarning`. Below: build at a compatible size (default cell 3) vs an incompatible size, and visualise the per-j seam-step ratio.

In [ ]:
import warnings as _w

def _seam_step_ratio(m):
    """For each seam node, return cap_first_step / equator_first_step (south seam)."""
    sub_blocks = m.blocks[:4]
    cap_south  = m.blocks[5].nodes[:, :, 0, :]
    nc = cap_south.shape[0]
    sector = nc - 1

    def colat(p):
        return float(np.arccos(np.clip(p[2] / np.linalg.norm(p), -1.0, 1.0)))

    ratios = []
    for jg in range(4 * sector):
        q, k = jg // sector, jg % sector
        # Equator side: i=0 (seam) to i=1 (one step inward, away from south pole).
        eq = abs(colat(sub_blocks[q].nodes[1, k, 0, :])
                 - colat(sub_blocks[q].nodes[0, k, 0, :]))
        # Cap side: seam node and its inward neighbour (closer to south pole).
        if   q == 0: seam, inw = cap_south[k, 0],            cap_south[k, 1]
        elif q == 1: seam, inw = cap_south[-1, k],           cap_south[-2, k]
        elif q == 2: seam, inw = cap_south[nc - 1 - k, -1],  cap_south[nc - 1 - k, -2]
        else:        seam, inw = cap_south[0, nc - 1 - k],   cap_south[1, nc - 1 - k]
        cap_step = colat(seam) - colat(inw)
        ratios.append(cap_step / eq if eq > 0 else np.nan)
    return np.array(ratios)

# Compatible sizes: default mesh (already built in cell 3).  No fallback warning.
m_compat = Mesh.from_stl(
    STL_PATH, topology=CubedSphereTopology(),
    ni=NI, nj=NJ, nk=3,
    theta_cap=THETA_CAP,
    spacing_k=Spacing1D(law='geometric', ds=DS, growth=GROWTH),
)
ratios_compat = _seam_step_ratio(m_compat)

# Incompatible sizes: theta_cap=9 deg with same ni/nj forces fallback.
with _w.catch_warnings(record=True) as caught:
    _w.simplefilter('always')
    m_incompat = Mesh.from_stl(
        STL_PATH, topology=CubedSphereTopology(),
        ni=NI, nj=NJ, nk=3,
        theta_cap=np.deg2rad(9.0),
        spacing_k=Spacing1D(law='geometric', ds=DS, growth=GROWTH),
    )
    fallback_warned = any('incompatible' in str(w.message) for w in caught)
ratios_incompat = _seam_step_ratio(m_incompat)

print(f'Compatible (theta_cap=30°): cap/eq step ratio = '
      f'{ratios_compat.mean():.3f} +/- {ratios_compat.std():.3f}  '
      f'(should be ~1.0)')
print(f'Incompatible (theta_cap=9°): cap/eq step ratio = '
      f'{ratios_incompat.mean():.3f} +/- {ratios_incompat.std():.3f}  '
      f'(unpinned fallback: {fallback_warned})')

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(ratios_compat,   'o-', ms=3, lw=1.2, label=f'compatible (theta_cap=30 deg)')
ax.plot(ratios_incompat, 's-', ms=3, lw=1.2, label=f'incompatible (theta_cap=9 deg, unpinned)')
ax.axhline(1.0, color='red', lw=0.8, ls='--', label='exact C1 match')
ax.set_xlabel('seam vertex j')
ax.set_ylabel('cap_step / equator_step')
ax.set_title('Per-j seam-step ratio: pinned vs unpinned')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11 — Volume seam continuity

The cap-equator C1 seam match (section 10) holds at `k=0` by construction.
For the spacing to stay continuous **at every wall-normal layer**, each
block's normal computation has to use centred differences across the
multi-block seams (a one-sided difference internal to each block would
disagree with the partner block's one-sided difference at the shared
seam, and the disagreement accumulates over many marching layers).

SHORE wires "ghost rows" between adjacent blocks at every layer of the
march so the boundary tangent uses a centred difference against the
partner's row-one-inward data. Below we measure the j-step ratio across
the sub0/sub1 seam at each layer and confirm the ratio stays close to
1.0 (it actually **converges** toward 1.0 as marching proceeds — the
centred differences damp small initial disagreements).

In [ ]:
m_vol = Mesh.from_stl(
    STL_PATH,
    topology=CubedSphereTopology(),
    ni=NI, nj=NJ, nk=20,
    theta_cap=THETA_CAP,
    spacing_k=Spacing1D(law='geometric', ds=DS, growth=GROWTH),
)
m_vol.march(ds=DS, growth=GROWTH)

ks = list(range(0, 20, 2))
ratios_per_k = []
for k in ks:
    sub0 = m_vol.blocks[0].nodes[:, :, k, :]
    sub1 = m_vol.blocks[1].nodes[:, :, k, :]
    s_left  = np.linalg.norm(sub0[:, -1, :] - sub0[:, -2, :], axis=-1)
    s_right = np.linalg.norm(sub1[:, 1,  :] - sub1[:, 0,  :], axis=-1)
    ratios_per_k.append(s_right / np.maximum(s_left, 1e-30))

fig, ax = plt.subplots(figsize=(9, 3))
for k, r in zip(ks, ratios_per_k):
    ax.plot(r, '-', lw=1.0, alpha=0.7, label=f'k={k}')
ax.axhline(1.0, color='red', lw=0.8, ls='--', label='exact match')
ax.set_xlabel('seam node i (sub0/sub1 j-seam)')
ax.set_ylabel('j-step ratio (sub1 first / sub0 last)')
ax.set_title('Sub-sub seam spacing ratio per marching layer')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('layer | max |ratio - 1|')
for k, r in zip(ks, ratios_per_k):
    print(f'  k={k:2d} | {np.max(np.abs(r - 1)):.4f}')